In [7]:
import asyncio
import tkinter as tk
from tkinter import ttk, scrolledtext, messagebox
from bleak import BleakScanner, BleakClient
import threading
import queue
from datetime import datetime

class BluetoothManager:
    def __init__(self, root):
        self.root = root
        self.root.title("Bluetooth Scanner & Connector")
        self.root.geometry("800x600")
        
        # Переменные для управления
        self.scanning = False
        self.connected_device = None
        self.client = None
        self.message_queue = queue.Queue()
        
        # Настройка UI
        self.setup_ui()
        
        # Запускаем обработчик очереди
        self.check_queue()
        
    def setup_ui(self):
        """Создание интерфейса"""
        # Верхняя панель с кнопками
        top_frame = tk.Frame(self.root)
        top_frame.pack(pady=10, fill=tk.X)
        
        self.btn_scan = tk.Button(top_frame, text="🔍 Начать сканирование", 
                                 command=self.start_scan, width=20, height=2)
        self.btn_scan.pack(side=tk.LEFT, padx=5)
        
        self.btn_stop = tk.Button(top_frame, text="⏹ Остановить", 
                                 command=self.stop_scan, width=20, height=2,
                                 state=tk.DISABLED)
        self.btn_stop.pack(side=tk.LEFT, padx=5)
        
        self.btn_disconnect = tk.Button(top_frame, text="🔌 Отключиться", 
                                       command=self.disconnect_device, width=20, height=2,
                                       state=tk.DISABLED)
        self.btn_disconnect.pack(side=tk.LEFT, padx=5)
        
        # Статус
        self.status_label = tk.Label(self.root, text="Готов к сканированию", 
                                     font=("Arial", 10), fg="blue")
        self.status_label.pack(pady=5)
        
        # Список устройств с прокруткой
        list_frame = tk.LabelFrame(self.root, text="📱 Найденные устройства", padx=5, pady=5)
        list_frame.pack(pady=10, padx=10, fill=tk.BOTH, expand=True)
        
        # Создаем Canvas для прокрутки
        canvas = tk.Canvas(list_frame)
        scrollbar = tk.Scrollbar(list_frame, orient="vertical", command=canvas.yview)
        self.scrollable_frame = tk.Frame(canvas)
        
        self.scrollable_frame.bind(
            "<Configure>",
            lambda e: canvas.configure(scrollregion=canvas.bbox("all"))
        )
        
        canvas.create_window((0, 0), window=self.scrollable_frame, anchor="nw")
        canvas.configure(yscrollcommand=scrollbar.set)
        
        canvas.pack(side="left", fill="both", expand=True)
        scrollbar.pack(side="right", fill="y")
        
        # Заголовки списка
        header_frame = tk.Frame(self.scrollable_frame)
        header_frame.pack(fill=tk.X, pady=2)
        tk.Label(header_frame, text="Устройство", width=25, anchor="w", font=("Arial", 10, "bold")).pack(side=tk.LEFT, padx=5)
        tk.Label(header_frame, text="MAC-адрес", width=20, anchor="w", font=("Arial", 10, "bold")).pack(side=tk.LEFT, padx=5)
        tk.Label(header_frame, text="RSSI", width=10, anchor="w", font=("Arial", 10, "bold")).pack(side=tk.LEFT, padx=5)
        tk.Label(header_frame, text="Действие", width=15, anchor="w", font=("Arial", 10, "bold")).pack(side=tk.LEFT, padx=5)
        
        # Словарь для хранения виджетов устройств
        self.device_widgets = {}
        
        # Лог сообщений
        log_frame = tk.LabelFrame(self.root, text="📝 Лог", padx=5, pady=5)
        log_frame.pack(pady=10, padx=10, fill=tk.BOTH, expand=True)
        
        self.log_text = scrolledtext.ScrolledText(log_frame, height=8, font=("Courier", 9))
        self.log_text.pack(fill=tk.BOTH, expand=True)
        
    def log_message(self, msg, color="black"):
        """Добавление сообщения в лог"""
        timestamp = datetime.now().strftime("%H:%M:%S")
        self.message_queue.put(f"[{timestamp}] {msg}|{color}")
        
    def check_queue(self):
        """Обработка очереди сообщений"""
        try:
            while True:
                msg = self.message_queue.get_nowait()
                if "|" in msg:
                    text, color = msg.split("|", 1)
                    self.log_text.insert(tk.END, text + "\n", color)
                    self.log_text.tag_config(color, foreground=color)
                else:
                    self.log_text.insert(tk.END, msg + "\n")
                self.log_text.see(tk.END)
        except queue.Empty:
            pass
        self.root.after(100, self.check_queue)
        
    def update_status(self, text, color="blue"):
        """Обновление статуса"""
        self.status_label.config(text=text, fg=color)
        
    def start_scan(self):
        """Запуск сканирования"""
        if self.scanning:
            return
            
        # Очищаем список устройств
        for widget in self.scrollable_frame.winfo_children():
            if widget != self.scrollable_frame.winfo_children()[0]:  # Оставляем заголовок
                widget.destroy()
        self.device_widgets.clear()
        
        self.scanning = True
        self.btn_scan.config(state=tk.DISABLED)
        self.btn_stop.config(state=tk.NORMAL)
        self.update_status("🔍 Идет сканирование...", "orange")
        self.log_message("Начало сканирования BLE устройств", "blue")
        
        # Запускаем сканирование в потоке
        threading.Thread(target=self.run_scan, daemon=True).start()
        
    def stop_scan(self):
        """Остановка сканирования"""
        self.scanning = False
        self.btn_stop.config(state=tk.DISABLED)
        self.update_status("Сканирование остановлено", "red")
        self.log_message("Сканирование остановлено пользователем", "red")
        
    def run_scan(self):
        """Запуск асинхронного сканирования"""
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        loop.run_until_complete(self.async_scan())
        loop.close()
        
        self.root.after(0, lambda: self.btn_scan.config(state=tk.NORMAL))
        self.root.after(0, lambda: self.btn_stop.config(state=tk.DISABLED))
        
    async def async_scan(self):
        """Асинхронное сканирование с callback"""
        try:
            # Создаем сканер с callback-функцией
            scanner = BleakScanner(self.device_found)
            await scanner.start()
            
            # Сканируем пока не остановят или не пройдет 10 секунд
            for _ in range(10):  # 10 секунд
                if not self.scanning:
                    break
                await asyncio.sleep(1)
            
            await scanner.stop()
            self.scanning = False
            
            if not self.scanning:  # Если не остановили вручную
                self.root.after(0, lambda: self.update_status("✅ Сканирование завершено", "green"))
                self.root.after(0, lambda: self.log_message("Сканирование завершено", "green"))
                
        except Exception as e:
            error_msg = str(e)
            self.root.after(0, lambda: self.log_message(f"Ошибка сканирования: {error_msg}", "red"))
            
        finally:
            self.root.after(0, lambda: self.btn_scan.config(state=tk.NORMAL))
            self.root.after(0, lambda: self.btn_stop.config(state=tk.DISABLED))
            
    def device_found(self, device, advertisement_data):
        """Callback при обнаружении устройства"""
        # Получаем RSSI из advertisement_data
        rssi = advertisement_data.rssi if hasattr(advertisement_data, 'rssi') else 0
        
        if device.address in self.device_widgets:
            # Обновляем RSSI
            rssi_label = self.device_widgets[device.address]["rssi"]
            rssi_label.config(text=f"{rssi} dBm")
            return
            
        # Создаем строку для нового устройства
        device_frame = tk.Frame(self.scrollable_frame)
        device_frame.pack(fill=tk.X, pady=2)
        
        # Имя устройства
        name = device.name or "Без имени"
        name_label = tk.Label(device_frame, text=name, width=25, anchor="w")
        name_label.pack(side=tk.LEFT, padx=5)
        
        # MAC-адрес
        mac_label = tk.Label(device_frame, text=device.address, width=20, anchor="w")
        mac_label.pack(side=tk.LEFT, padx=5)
        
        # RSSI
        rssi_label = tk.Label(device_frame, text=f"{rssi} dBm", width=10, anchor="w")
        rssi_label.pack(side=tk.LEFT, padx=5)
        
        # Кнопка подключения
        connect_btn = tk.Button(device_frame, text="🔗 Подключиться", 
                               command=lambda addr=device.address, name=name: self.connect_device(addr, name),
                               width=15)
        connect_btn.pack(side=tk.LEFT, padx=5)
        
        # Сохраняем ссылки
        self.device_widgets[device.address] = {
            "frame": device_frame,
            "name": name_label,
            "mac": mac_label,
            "rssi": rssi_label,
            "button": connect_btn
        }
        
        self.root.after(0, lambda: self.log_message(f"Найдено: {name} ({device.address}) RSSI: {rssi} dBm", "green"))
        
    def connect_device(self, address, name):
        """Подключение к устройству"""
        if self.connected_device:
            messagebox.showwarning("Подключение", f"Уже подключены к {self.connected_device}")
            return
            
        self.update_status(f"🔄 Подключение к {name}...", "orange")
        self.log_message(f"Попытка подключения к {name} ({address})", "blue")
        
        # Запускаем подключение в потоке
        threading.Thread(target=self.run_connect, args=(address, name), daemon=True).start()
        
    def run_connect(self, address, name):
        """Подключение в отдельном потоке"""
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        loop.run_until_complete(self.async_connect(address, name))
        loop.close()
        
    async def async_connect(self, address, name):
        """Асинхронное подключение к устройству"""
        try:
            self.client = BleakClient(address)
            await self.client.connect()
            
            # Сохраняем информацию о подключении
            self.connected_device = name
            
            self.root.after(0, lambda: self.update_status(f"✅ Подключено к {name}", "green"))
            self.root.after(0, lambda: self.log_message(f"✅ Успешно подключено к {name}", "green"))
            self.root.after(0, lambda: self.btn_disconnect.config(state=tk.NORMAL))
            
            # Блокируем кнопки подключения
            for addr, widgets in self.device_widgets.items():
                if addr == address:
                    self.root.after(0, lambda w=widgets: w["button"].config(text="✅ Подключено", state=tk.DISABLED, bg="lightgreen"))
                else:
                    self.root.after(0, lambda w=widgets: w["button"].config(state=tk.DISABLED))
            
            # Показываем информацию об устройстве
            await self.show_device_info()
            
            # Демонстрация получения данных
            await self.demo_data_exchange()
            
        except Exception as e:
            error_msg = str(e)
            self.root.after(0, lambda: self.update_status(f"❌ Ошибка подключения", "red"))
            self.root.after(0, lambda: self.log_message(f"❌ Ошибка подключения: {error_msg}", "red"))
            self.root.after(0, lambda: messagebox.showerror("Ошибка", f"Не удалось подключиться:\n{error_msg}"))
            
    async def show_device_info(self):
        """Показываем информацию об устройстве - ИСПРАВЛЕНАЯ ВЕРСИЯ"""
        if not self.client or not self.client.is_connected:
            return
            
        try:
            # Получаем список сервисов через свойство services
            services = await self.client.get_services()
            
            # Альтернативный способ (для новых версий bleak):
            # services = self.client.services
            
            self.root.after(0, lambda: self.log_message(f"📋 Найдено сервисов: {len(services)}", "blue"))
            
            for service in services:
                self.root.after(0, lambda s=service: self.log_message(f"  • Сервис: {s.uuid}", "black"))
                for char in service.characteristics:
                    self.root.after(0, lambda c=char: self.log_message(f"    - Характеристика: {c.uuid}", "gray"))
                    
        except AttributeError:
            # Если метод get_services() не работает, пробуем другой способ
            try:
                services = self.client.services
                self.root.after(0, lambda: self.log_message(f"📋 Найдено сервисов: {len(services)}", "blue"))
                
                for service in services:
                    self.root.after(0, lambda s=service: self.log_message(f"  • Сервис: {s.uuid}", "black"))
                    for char in service.characteristics:
                        self.root.after(0, lambda c=char: self.log_message(f"    - Характеристика: {c.uuid}", "gray"))
            except Exception as e:
                self.root.after(0, lambda: self.log_message(f"Ошибка получения информации: {str(e)}", "red"))
        except Exception as e:
            self.root.after(0, lambda: self.log_message(f"Ошибка получения информации: {str(e)}", "red"))
            
    async def demo_data_exchange(self):
        """Демонстрация обмена данными - ИСПРАВЛЕННАЯ ВЕРСИЯ"""
        if not self.client or not self.client.is_connected:
            return
            
        try:
            # Получаем сервисы
            try:
                services = await self.client.get_services()
            except AttributeError:
                services = self.client.services
                
            for service in services:
                for char in service.characteristics:
                    if "read" in char.properties:
                        try:
                            value = await self.client.read_gatt_char(char.uuid)
                            self.root.after(0, lambda: self.log_message(f"📖 Прочитано: {char.uuid} = {value[:20] if value else 'None'}...", "green"))
                        except:
                            pass
        except Exception as e:
            self.root.after(0, lambda: self.log_message(f"Ошибка чтения: {str(e)}", "red"))
            
    def disconnect_device(self):
        """Отключение от устройства"""
        if not self.connected_device:
            return
            
        self.update_status("🔄 Отключение...", "orange")
        self.log_message(f"Отключение от {self.connected_device}", "blue")
        
        # Запускаем отключение в потоке
        threading.Thread(target=self.run_disconnect, daemon=True).start()
        
    def run_disconnect(self):
        """Отключение в отдельном потоке"""
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        loop.run_until_complete(self.async_disconnect())
        loop.close()
        
    async def async_disconnect(self):
        """Асинхронное отключение"""
        try:
            if self.client and self.client.is_connected:
                await self.client.disconnect()
                
            self.root.after(0, lambda: self.update_status("Отключено", "blue"))
            self.root.after(0, lambda: self.log_message("✅ Отключено от устройства", "green"))
            self.root.after(0, lambda: self.btn_disconnect.config(state=tk.DISABLED))
            
            # Возвращаем кнопки подключения в активное состояние
            for widgets in self.device_widgets.values():
                self.root.after(0, lambda w=widgets: w["button"].config(text="🔗 Подключиться", state=tk.NORMAL, bg="SystemButtonFace"))
                
            self.connected_device = None
            self.client = None
            
        except Exception as e:
            self.root.after(0, lambda: self.log_message(f"❌ Ошибка отключения: {str(e)}", "red"))
            
    def on_closing(self):
        """Закрытие приложения"""
        if self.client and self.client.is_connected:
            if messagebox.askokcancel("Выход", "Устройство подключено. Отключиться?"):
                self.disconnect_device()
                self.root.destroy()
        else:
            self.root.destroy()

if __name__ == "__main__":
    root = tk.Tk()
    app = BluetoothManager(root)
    root.protocol("WM_DELETE_WINDOW", app.on_closing)
    root.mainloop()

Exception in Tkinter callback
Traceback (most recent call last):
  File "D:\Python313\Lib\tkinter\__init__.py", line 2074, in __call__
    return self.func(*args)
           ~~~~~~~~~^^^^^^^
  File "D:\Python313\Lib\tkinter\__init__.py", line 862, in callit
    func(*args)
    ~~~~^^^^^^^
  File "C:\Users\Iziaslaw\AppData\Local\Temp\ipykernel_14128\2289097916.py", line 299, in <lambda>
    self.root.after(0, lambda: self.log_message(f"📋 Найдено сервисов: {len(services)}", "blue"))
                                                                        ~~~^^^^^^^^^^
TypeError: object of type 'BleakGATTServiceCollection' has no len()
